<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/Kimi_K3_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.kimi.com/chat/19a5ed9a-85c2-8c42-8000-0957e3e579bc

In [3]:
from openai import OpenAI
from google.colab import userdata
import time
import json
from pathlib import Path
from datetime import datetime

# ============================================
# CONFIGURATION
# ============================================
MODEL_NAME = "kimi-k3"
SYSTEM_PROMPT = """You are Kimi K3, an expert in theoretical physics, computer science, and complex reasoning.
Your task is to generate a detailed, speculative, and coherent hypothesis for the user's question.
Be rigorous, cite relevant work where appropriate, and explicitly flag speculative elements."""

API_KEY = userdata.get('KIMI_API_KEY')
BASE_URL = "https://api.moonshot.ai/v1"
OUTPUT_TOKEN_PRICE = 15.0  # $ per million tokens

# ============================================
# THE THREE DIFFICULT QUESTIONS
# ============================================
COMPLEX_QUESTIONS = [
    (
        "If the P ≠ NP problem were definitively proven true, how might this computational "
        "limit offer a fundamental constraint or insight into solving the problem of Quantum "
        "Gravity or determining the underlying nature of Dark Matter? Articulate a speculative, "
        "testable hypothesis connecting the theoretical limits of computation to the deepest "
        "mysteries of the physical universe."
    ),
    (
        "The 'Hard Problem of Consciousness' asks why physical processes create subjective experience. "
        "The 'AI Alignment Problem' asks how we encode human values into a machine. If we assume that "
        "successful alignment requires an AI to model or possess a form of sentience: Which core "
        "philosophical stance on consciousness (e.g., Integrated Information Theory, Global Workspace "
        "Theory, Panpsychism) offers the most pragmatic path forward for solving the AI Alignment "
        "problem, and why?"
    ),
    (
        "The Black Hole Information Paradox, the P vs. NP problem, and the question of how terminal "
        "values are encoded in an LLM (AI Alignment) all fundamentally deal with the creation, retention, "
        "and complexity of information. Formulate a single, overarching principle related to 'The "
        "Absolute Limit of Information Density and Complexity' that could potentially resolve one "
        "paradox in physics, one complexity class question in computer science, and one challenge in "
        "AI alignment."
    )
]

# ============================================
# HELPER FUNCTIONS
# ============================================

def estimate_cost(output_tokens):
    return (output_tokens / 1_000_000) * OUTPUT_TOKEN_PRICE

def format_time(seconds):
    if seconds < 60:
        return f"{seconds:.1f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        return f"{seconds/3600:.1f}h"

def save_response(question_idx, response_data):
    cache_dir = Path("kimi_responses")
    cache_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = cache_dir / f"response_q{question_idx+1}_{timestamp}.json"
    with open(filename, 'w') as f:
        json.dump(response_data, f, indent=2)
    return filename

def load_cached_response(question_idx):
    cache_dir = Path("kimi_responses")
    if not cache_dir.exists():
        return None
    files = sorted(cache_dir.glob(f"response_q{question_idx+1}_*.json"), reverse=True)
    if files:
        with open(files[0]) as f:
            return json.load(f)
    return None

def count_tokens_estimate(text):
    """Estimate tokens: ~4 characters per token for English text."""
    if not text:
        return 0
    # Rough but reasonable estimate
    return max(1, len(text) // 4)

# ============================================
# MAIN - STREAMING WITH TOKEN COUNTING
# ============================================

def main():
    print("="*60)
    print("🚀 KIMI K3 DEMO - Complex Reasoning")
    print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"🤖 Model: {MODEL_NAME}")
    print(f"⚙️  Reasoning Effort: max")
    print("="*60)

    if not API_KEY:
        print("❌ FATAL: KIMI_API_KEY not found in Colab secrets.")
        return

    # Client with generous timeout
    client = OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
        timeout=600.0  # 10 minutes
    )

    results = []
    total_cost = 0.0
    total_output_tokens = 0
    total_time = 0

    for i, question in enumerate(COMPLEX_QUESTIONS):
        print(f"\n{'='*60}")
        print(f"🧠 QUESTION {i+1} of {len(COMPLEX_QUESTIONS)}")
        print(f"📝 {question[:100]}...")
        print(f"{'='*60}")

        # Check cache
        cached = load_cached_response(i)
        if cached:
            print("\n📂 Using cached response...")
            results.append(cached)
            total_cost += cached.get('cost_estimate', 0)
            total_output_tokens += cached.get('total_tokens_estimate', 0)
            total_time += cached.get('elapsed_seconds', 0)
            continue

        start_time = time.time()

        try:
            print("\n⏳ Generating response (streaming)...")
            print("   (Reasoning will appear as it's generated)\n")

            # Streaming request
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": question}
                ],
                stream=True,
                reasoning_effort="max"
            )

            reasoning_text = []
            final_text = []
            thinking_complete = False

            print("--- Internal Reasoning Content (Thinking Trace) ---")

            # Collect all chunks
            for chunk in response:
                delta = chunk.choices[0].delta

                reasoning = getattr(delta, "reasoning_content", None)
                if reasoning:
                    reasoning_text.append(reasoning)
                    print(reasoning, end="", flush=True)

                content = getattr(delta, "content", None)
                if content:
                    if not thinking_complete:
                        print("\n\n--- Kimi K3 Final Answer ---")
                        thinking_complete = True
                    final_text.append(content)
                    print(content, end="", flush=True)

            print("\n")

            elapsed = time.time() - start_time

            # Combine texts
            reasoning_full = "".join(reasoning_text)
            final_full = "".join(final_text)

            # Count tokens from actual response
            reasoning_tokens = count_tokens_estimate(reasoning_full)
            final_tokens = count_tokens_estimate(final_full)
            total_tokens = reasoning_tokens + final_tokens
            cost_estimate = estimate_cost(total_tokens)

            # Build result with token counts
            result = {
                "reasoning": reasoning_full,
                "final_answer": final_full,
                "reasoning_tokens": reasoning_tokens,
                "final_tokens": final_tokens,
                "total_tokens_estimate": total_tokens,
                "cost_estimate": cost_estimate,
                "elapsed_seconds": elapsed,
                "timestamp": datetime.now().isoformat(),
                "question": question,
                "mode": "streaming"
            }

            # Save cache
            save_response(i, result)
            results.append(result)

            total_cost += cost_estimate
            total_output_tokens += total_tokens
            total_time += elapsed

            # Display stats with token counts
            print(f"\n📊 Stats:")
            print(f"   ⏱️  Time: {format_time(elapsed)}")
            print(f"   📝 Reasoning tokens: {reasoning_tokens:,}")
            print(f"   📝 Final tokens:     {final_tokens:,}")
            print(f"   📝 Total tokens:     {total_tokens:,}")
            print(f"   💰 Est. cost:        ${cost_estimate:.4f}")

        except Exception as e:
            print(f"\n❌ ERROR on Question {i+1}: {e}")
            continue

    # ============================================
    # SUMMARY REPORT
    # ============================================
    print("\n" + "="*60)
    print("📊 SUMMARY REPORT")
    print("="*60)

    print(f"✅ Questions completed: {len(results)}/{len(COMPLEX_QUESTIONS)}")
    print(f"📝 Total output tokens: {total_output_tokens:,}")
    print(f"💰 Estimated total cost: ${total_cost:.4f}")
    print(f"⏱️  Total time: {format_time(total_time)}")

    print("\n📋 Per-Question Breakdown:")
    for i, result in enumerate(results):
        tokens = result.get('total_tokens_estimate', 0)
        cost = result.get('cost_estimate', 0)
        elapsed = result.get('elapsed_seconds', 0)
        print(f"   Q{i+1}: {tokens:>6,} tokens | ${cost:.4f} | {format_time(elapsed)}")

    print("\n" + "="*60)
    print("🎉 Done! Responses saved to kimi_responses/")
    print("="*60)

if __name__ == "__main__":
    main()

🚀 KIMI K3 DEMO - Complex Reasoning
📅 2026-07-18 14:32:07
🤖 Model: kimi-k3
⚙️  Reasoning Effort: max

🧠 QUESTION 1 of 3
📝 If the P ≠ NP problem were definitively proven true, how might this computational limit offer a fund...

⏳ Generating response (streaming)...
   (Reasoning will appear as it's generated)

--- Internal Reasoning Content (Thinking Trace) ---
The user is asking a deeply speculative question connecting computational complexity theory (P ≠ NP) to quantum gravity and dark matter. This is a fascinating interdisciplinary question that requires me to:

1. Be rigorous about what's known
2. Be speculative but coherent
3. Cite relevant work
4. Propose a testable hypothesis
5. Flag speculative elements clearly

Let me think through the actual intellectual landscape here:

**Background on P vs NP and physics connections:**

1. **Scott Aaronson's work**: He's written extensively on complexity theory and physics. His paper "NP-complete Problems and Physical Reality" (2005) discusses